## Web scrapper to download all papers related to SO data

In [2]:
import time, os, requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display, HTML
import re, string, unicodedata, wget
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

base_dir = os.getcwd()

In [2]:
url = 'https://meta.stackexchange.com/questions/134495/academic-papers-using-stack-exchange-data'
page = requests.get(url)
session = requests.Session()
page = session.get(url)
soup = BeautifulSoup(page.content, 'html.parser')

In [3]:
all_papers = soup.find_all(class_ = 's-prose js-post-body')

In [4]:

for i in range(1, len(all_papers)):
    h3s = all_papers[i].find_all('h3')
    for h in h3s:
        folder_name = h.text
        if not os.path.exists(str(folder_name)):
            os.makedirs(str(folder_name))

In [5]:
papers_all_links = pd.DataFrame(columns = ['paper_name', 'available_link', 'year', 'pdf_availability'])

In [6]:
#to save all papers' data
papers_all_links = pd.DataFrame(columns = ['paper_name', 'available_link', 'year', 'pdf_availability'])
valid_files = pd.DataFrame(columns = ['file', 'local_path'])
for i in range(1, len(all_papers)):
    for elem in all_papers[i]:
        if(elem.name=='h3'):
            year = elem.text
        elif(elem.name=='ul'):
            list_items = elem.find_all('li')
            for item in list_items:
                t = item.text
                t = re.sub('\[.*\]', '', t)
                t = re.sub('\\n', ' ', t)
                t = re.sub('\.\s*', '. ', t)
                paper_name = t
                #------
                links = item.find_all('a')
                for link in links:
                    current_link = link.get('href')
                    if current_link.lower().endswith('pdf'):
                        flag = 'Yes'
                        #download and save pdf to year_dir
                        
                        paper_name = re.sub('[^A-Za-z0-9]', '_', paper_name[:100])
                        local_path = os.path.join(year, paper_name + '.pdf')
                        down_dir = os.path.join(base_dir, local_path)                                
                        
                        try:                            
                            print('\n\n')
                            print('Downloading: ', current_link, '\n', down_dir)
                            r = requests.get(current_link)
                            with open(down_dir, 'wb') as f:
                                f.write(r.content)
                                size = os.path.getsize(down_dir)
                                
                                if size > 0:
                                    if (size / 1024) > 15:
                                        print('Found')
                                        file_data = {
                                            'file': paper_name,
                                            'local_path': local_path
                                        }                                    
                                        valid_files = valid_files.append(file_data, ignore_index=True)                                    
                            print("Saved to {}".format(down_dir))
                        except Exception as e:
                            print("Downloading Failed: " + current_link)
                            print(str(e) + "\n\n")
                            
                        #print(paper_name, current_link, year, 'Yes')
                    else:
                        flag = 'No'
                        #can not be downloaded
                        #print(paper_name, current_link, year, 'No')
                    #saved to a dataframe, later CSV
                    
                    temp = {'paper_name':paper_name,
                                'available_link':current_link,
                                'year':year,
                                'pdf_availability': flag} 
                    #print(temp)
                    papers_all_links = papers_all_links.append(temp, ignore_index=True)
                    #print(temp)
                    
papers_all_links.to_csv(os.path.join(base_dir, 'papers_all_links.csv')
valid_files.to_csv(os.path.join(base_dir, 'valid_files.csv'))
print('Saved...', file_path)




Downloading:  https://arxiv.org/pdf/2010.04025.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2020/Timur_Bachschi__Aniko_Hannak__Florian_Lemmerich__and_Johannes_Wachs__From_Asking_to_Answering__Getti.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2020/Timur_Bachschi__Aniko_Hannak__Florian_Lemmerich__and_Johannes_Wachs__From_Asking_to_Answering__Getti.pdf



Downloading:  http://empirical-software.engineering/assets/pdf/tse19-condor.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2020/Sebastian_Baltes__Christoph_Treude__and_Martin_Robillard__Contextual_Documentation_Referencing_on_St.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2020/Sebastian_Baltes__Christoph_Treude__and_Martin_Robillard__Contextual_Documentation_Referencing_on_St.pdf



Downloading:  http://empirical-software.engineering/assets/pdf/icse20-clones.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/202

Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2018/Sebastian_Baltes_and_Stephan_Diehl__Usage_and_Attribution_of_Stack_Overflow_Code_Snippets_in_GitHub_.pdf



Downloading:  https://chunyang-chen.github.io/publication/diffSimilarTech.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2018/Yi_Huang__Chunyang_Chen__Zhenchang_Xing__Tian_Lin_and_Yang_Liu__Tell_Them_Apart__Distilling_Technolo.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2018/Yi_Huang__Chunyang_Chen__Zhenchang_Xing__Tian_Lin_and_Yang_Liu__Tell_Them_Apart__Distilling_Technolo.pdf



Downloading:  http://sail.cs.queensu.ca/Downloads/EMSE2018_HowDoDevelopersUtilizeSourceCodeFromStackOverflow.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2018/Yuhao_Wu__Shaowei_Wang_Cor_Paul_Bezemer__Katsuro_Inoue__How_Do_Developers_Utilize_Source_Code_from_S.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2018/Yuhao_Wu_

Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2016/Boudaer__Glenn__and_Johan_Loeckx__Enriching_Topic_Modelling_with_Users__Histories_for_Improving_Tag_.pdf



Downloading:  http://lascam.facom.ufu.br/cms/userfiles/downloads/2016/cascon-preprint.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2016/Eduardo_Campos__Martin_Monperrus__Marcelo_Maia__Searching_Stack_Overflow_for_API_usage_related_Bug_F.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2016/Eduardo_Campos__Martin_Monperrus__Marcelo_Maia__Searching_Stack_Overflow_for_API_usage_related_Bug_F.pdf



Downloading:  http://lascam.facom.ufu.br/cms/userfiles/downloads/2016/jsep2016AcceptedVersion.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2016/Eduardo_Campos__Lucas_Souza__Marcelo_Maia__Searching_Crowd_Knowledge_to_Recommend_Solutions_for_API_.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2016/Eduardo_Campos__Lucas_Souza__M

Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2015/Luca_Ponzanelli__Andrea_Mocci__Michele_Lanza__StORMeD__Stack_Overflow_Ready_Made_Data__In_Proceeding.pdf



Downloading:  https://dl.dropboxusercontent.com/u/3463652/SSE_Q%26A_CameraReady.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2015/Nicole_Novielli__Fabio_Calefato__Filippo_Lanubile__The_Challenges_of_Sentiment_Detection_in_the_Soci.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2015/Nicole_Novielli__Fabio_Calefato__Filippo_Lanubile__The_Challenges_of_Sentiment_Detection_in_the_Soci.pdf



Downloading:  http://www.cs.cmu.edu/~jssunshi/pubs/icpc15-searching.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2015/Joshua_Sunshine__James_D__Herbsleb__and_Jonathan_Aldrich__Searching_the_State_Space__A_Qualitative_S.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2015/Joshua_Sunshine__James_D__Herbsleb__and_Jonathan_Aldrich

Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2014/Megan_Squire_and_Christian_Funkhouser__2014___A_Bit_of_Code___How_the_Stack_Overflow_Community_Creat.pdf



Downloading:  http://www.cs.cmu.edu/~ylataus/files/TausczikKitturKraut2014.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2014/Yla_Tausczik__Aniket_Kittur_and_Robert_Kraut__2014__Collaborative_problem_solving__A_study_of_Math_O.pdf
Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2014/Yla_Tausczik__Aniket_Kittur_and_Robert_Kraut__2014__Collaborative_problem_solving__A_study_of_Math_O.pdf



Downloading:  http://www.iiitd.edu.in/~ashish/APSEC2014SD.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2014/Sangeeta_Lal__Denzil_Correa__and_Ashish_Sureka__2014__MiQs__Characterization_and_Prediction_of_Migra.pdf
HTTPSConnectionPool(host='www.iiitd.edu.in', port=443): Max retries exceeded with url: /~ashish/APSEC2014SD.pdf (Caused by SSLError(SSLErro

Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2013/Muhammad_Asaduzzaman__Ahmed_Mashiyat__Chanchal_Roy_and_Kevin_Schneider__2013__Answering_Questions_Ab.pdf



Downloading:  http://cscorley.students.cs.ua.edu/_static/pubs/msr13-id184-p-15930-preprint.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2013/Amiangshu_Bosu__Christopher_Corley__Dustin_Heaton__Debarshi_Chatterji__Jeffrey_Carver_and_Nicholas_K.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2013/Amiangshu_Bosu__Christopher_Corley__Dustin_Heaton__Debarshi_Chatterji__Jeffrey_Carver_and_Nicholas_K.pdf



Downloading:  http://webdocs.cs.ualberta.ca/~joshua2/documentation.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2013/Joshua_Campbell__Chenlei_Zhang__Zhen_Xu__Abram_Hindle_and_James_Miller__2013__Deficient_Documentatio.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2013/Joshua_Campbell__Chenlei_Zhang__Zhen_Xu__Abram

Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2012/Aditya_Pal__Shuo_Chang__and_Joseph_A__Konstan__2012__Evolution_of_Experts_in_Question_Answering_Comm.pdf



Downloading:  http://www.cc.gatech.edu/~vector/papers/CrowdDoc-GIT-CS-12-05.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2012/Chris_Parnin__Christoph_Treude__Lars_Grammel__and_Margaret_Anne_Storey__2012__Crowd_documentation__E.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2012/Chris_Parnin__Christoph_Treude__Lars_Grammel__and_Margaret_Anne_Storey__2012__Crowd_documentation__E.pdf



Downloading:  http://dalspace.library.dal.ca/bitstream/handle/10222/14580/Riahi%2c%20Fatemeh%2c%20MSc%2c%20CS%2c%20February%202012.pdf 
 /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2012/Fatemeh_Riahi__2012__Finding_expert_users_in_community_question_answering_services_using_topic_model.pdf
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2012/Fat

Found
Saved to /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/2009/Erik_Wilde_Feeds_as_Query_Result_Serializations_In_arXiv_0911__2193__.pdf
Saved... /Users/antique/Documents/GitHub/Sites/so-papers-simillaritypapers_all_links.csv


In [54]:
base_dir = os.getcwd()
print(base_dir)
papers_all_links = pd.read_csv('papers_all_links.csv')
print('Unique papers#',len(papers_all_links['paper_name'].unique()))
print('PDF Available#', len(papers_all_links[papers_all_links['pdf_availability']=='Yes']))

unique_papers = pd.DataFrame(columns = ['paper_name', 'available_link', 'year', 'pdf_availability'])

for i in range(1, len(papers_all_links)):
    pn = papers_all_links.iloc[i-1]['paper_name']
    pp = papers_all_links.iloc[i-1]['pdf_availability']
    #print(pn[:20], pp)
    
    cn = papers_all_links.iloc[i]['paper_name']
    cp = papers_all_links.iloc[i]['pdf_availability']
    #print(cn[:20], cp)
    if(cn == pn):
        if(pp == 'No'):
            unique_papers.drop(unique_papers.tail(1).index, inplace = True)
            unique_papers = unique_papers.append(papers_all_links.iloc[i])
        else:
            pass
    else:
        unique_papers = unique_papers.append(papers_all_links.iloc[i])
        #print(papers_all_links.iloc[i]['paper_name'],'\n')
        
file_path = base_dir + '/unique_papers.csv'
unique_papers.to_csv(file_path)
print('Saved...', file_path)
print(unique_papers.shape)
unique_papers.head()

/Users/antique/Documents/GitHub/Sites/so-papers-simillarity
Unique papers# 189
PDF Available# 133
Saved... /Users/antique/Documents/GitHub/Sites/so-papers-simillarity/unique_papers.csv
(189, 5)


,paper_name,available_link,year,pdf_availability,Unnamed: 0
1,"Rodrigo Silva, Chanchal Roy, Masud Rahman, Kev...",https://doi.org/10.1007/s10664-020-09863-2,2020,No,1.0
2,"Preetha Chatterjee, Minji Kong, Lori Pollock. ...",https://doi.org/10.1016/j.jss.2019.110454,2020,No,2.0
3,"Saikat Mondal, Gias Uddin, and Chanchal K. Roy...",https://osf.io/sjgnz/,2020,No,3.0
4,"Timur Bachschi, Aniko Hannak, Florian Lemmeric...",https://arxiv.org/pdf/2010.04025.pdf,2020,Yes,4.0
5,"Sebastian Baltes, Christoph Treude, and Martin...",http://empirical-software.engineering/assets/p...,2020,Yes,5.0


## Extracting abstrct

In [13]:
from tika import parser
import fitz
def get_abstract(pdf_path):
    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += page.getText()
        text = str(text)
        abstract = re.search(r"Abstract(.*?)(Keyword|Index Terms|..\sINTRODUCTION|CCS Concept)", text, flags=re.DOTALL|re.I)
        abstract = abstract.group(1)
        abstract = re.sub(r'\s+', " ", abstract).strip()
        abstract = re.sub('^[^A-Za-z0-9]', "", abstract, flags=re.DOTALL)
        #print(abstract)
        return abstract 
    except:
        print("Error")

In [16]:
def extract_data(dataset, csv_path):
    df = pd.read_csv(csv_path)
    #files = df['local_path']
    for i in range(0, len(df)):
        fi = df.iloc[i]['file']
        lp = df.iloc[i]['local_path']
        print("\nExtracting Abstract: {}".format(fi))
        abstract = get_abstract(lp)
        temp = {'file': fi ,'local_path': lp, 'abstract': abstract}
        dataset = dataset.append(temp, ignore_index=True)
    
    dataset.to_csv(os.path.join(base_dir, 'dataset.csv'))
    print('Saved...dataset!!')

In [17]:
dataset = pd.DataFrame(columns = ['file', 'local_path', 'abstract'])
csv_path = 'valid_files.csv'
extract_data(dataset, csv_path)


Extracting Abstract: Timur_Bachschi__Aniko_Hannak__Florian_Lemmerich__and_Johannes_Wachs__From_Asking_to_Answering__Getti

Extracting Abstract: Sebastian_Baltes__Christoph_Treude__and_Martin_Robillard__Contextual_Documentation_Referencing_on_St

Extracting Abstract: Sebastian_Baltes_and_Christoph_Treude__Code_Duplication_on_Stack_Overflow__Proceedings_of_the_42nd_I

Extracting Abstract: Sebastian_Baltes_and_Markus_Wagner__An_Annotated_Dataset_of_Stack_Overflow_Post_Edits__Genetic_and_E

Extracting Abstract: Sengupta__S______Haythornthwaite__C__Learning_with_comments__An_analysis_of_comments_and_community_o

Extracting Abstract: Preetha_Chatterjee__Kostadin_Damevski__Lori_Pollock__Vinay_Augustine__and_Nicholas_A__Kraft__Explora

Extracting Abstract: Suyu_Ma__Zhenchang_Xing__Chunyang_Chen__Cheng_Chen__Lizhen_Qu__Guoqiang_Li__Easy_to_Deploy_API_Extra

Extracting Abstract: Xiang_Chen__Chunyang_Chen__Dun_Zhang__Zhenchang_Xing__SEthesaurus__WordNet_in_Software_Engineering__

Extracting Abst

mupdf: cannot recognize version marker
mupdf: no objects found
mupdf: cannot recognize version marker
mupdf: no objects found
mupdf: cannot recognize version marker
mupdf: no objects found
mupdf: cannot recognize version marker
mupdf: no objects found



Extracting Abstract: Luca_Ponzanelli__Gabriele_Bavota__Massimiliano_Di_Penta__Rocco_Oliveto__Michele_Lanza__2014__Prompte

Extracting Abstract: Shaowei_Wang__David_Lo__Bogdan_Vasilescu__and_Alexander_Serebrenik__2014__EnTagRec__An_Enhanced_Tag_
Error

Extracting Abstract: Bogdan_Vasilescu__Alexander_Serebrenik__Prem_Devanbu_and_Vladimir_Filkov__2014__How_social_Q_A_sites
Error

Extracting Abstract: Megan_Squire_and_Christian_Funkhouser__2014___A_Bit_of_Code___How_the_Stack_Overflow_Community_Creat

Extracting Abstract: Yla_Tausczik__Aniket_Kittur_and_Robert_Kraut__2014__Collaborative_problem_solving__A_study_of_Math_O

Extracting Abstract: Dana_Movshovitz_Attias_and_William_Cohen__2013__Natural_Language_Models_for_Predicting_Programming_C

Extracting Abstract: Galina_E__Lezina__Artem_M__Kuznetsov__2013__Predict_Closed_Questions_on_StackOver_ow_In_Proceedings_

Extracting Abstract: Dana_Movshovitz_Attias__Yair_Movshovitz_Attias__Peter_Steenkiste_and_Christos_Faloutsos__2013__Analy

Ext

mupdf: cannot recognize version marker
mupdf: no objects found
mupdf: cannot recognize version marker
mupdf: no objects found



Extracting Abstract: Carlos_Gomez__Brendan_Cleary_and_Leif_Singer__2013__A_Study_of_Innovation_Diffusion_Through_Link_Sha

Extracting Abstract: Scott_Grant_and_Buddy_Betts__2013__Encouraging_User_Behaviour_With_Achievements__An_Empirical_Study_

Extracting Abstract: Patrick_Morrison_and_Emerson_Murphy_Hill__2013__Is_Programming_Knowledge_Related_To_Age____An_Explor

Extracting Abstract: Siddharth_Subramanian_and_Reid_Holmes__2013__Making_Sense_of_Online_Code_Snippets_In_10th_Working_Co

Extracting Abstract: Wei_Wang_and_Michael_Godfrey__2013__Detecting_API_Usage_Obstacles__A_Study_of_iOS_and_Android_Develo

Extracting Abstract: Yla_Tausczik_and_James_Pennebaker__2012__Participation_in_an_online_mathematics_community__Different

Extracting Abstract: Leman_Akoglu__Duen_Horng_Chau__U__Kang__Danai_Koutra_and_Christos_Faloutsos__2012__OPAvion__mining_a
Error

Extracting Abstract: Ashton_Anderson__Daniel_Huttenlocher__Jon_Kleinberg_and_Jure_Leskovec__2012__Discovering_value_from_


mupdf: cannot recognize version marker
mupdf: no objects found



Extracting Abstract: Alberto_Bacchelli__Luca_Ponzanelli_and_Michele_Lanza__2012__Harnessing_Stack_Overflow_for_the_IDE__I

Extracting Abstract: Seyed_Mehdi_Nasehi__Jonathan_Sillito__Frank_Maurer_and_Chris_Burns__2012__What_Makes_a_Good_Code_Exa

Extracting Abstract: Leif_Singer__Fernando_Figueira_Filho__Brendan_Cleary__Christoph_Treude__Margaret_Anne_Storey__and_Ku

Extracting Abstract: Christoph_Treude__Fernando_Figueira_Filho__Brendan_Cleary__and_Margaret_Anne_Storey__2012__Programmi

Extracting Abstract: Bogdan_Vasilescu__Andrea_Capiluppi_and_Alexander_Serebrenik__2012__Gender__representation_and_online
Error

Extracting Abstract: Jakob_Vo___2012__Linking_Folksonomies_to_Knowledge_Organization_Systems__In_6th_Conference_on_Metada

Extracting Abstract: Alexey_Zagalsky__Ohad_Barzilay__and_Amiram_Yehudai__2012__Example_overflow__Using_social_media_for_c


mupdf: cannot recognize version marker
mupdf: no objects found



Extracting Abstract: Yla_Tausczik_and_James_Pennebaker__2011__Predicting_the_Perceived_Quality_of_Online_Mathematics_Cont

Extracting Abstract: Zainab_Zolaktaf__Fatemeh_Riahi__Mahdi_Shafiei__Evangelos_Milios__2011__Modeling_Community_Question_A

Extracting Abstract: Lena_Mamykina__Bella_Manoim__Manas_Mittal__George_Hripcsak__and_Bj_rn_Hartmann__2011__Design_lessons

Extracting Abstract: Zainab_Zolaktaf__Fatemeh_Riahi__Mahdi_Sha_ei__and_Evangelos_Milios__Modeling_community_question_answ

Extracting Abstract: Matthew_J__H__Rattigan_and_David_Jensen__2010__Leveraging_D_Separation_for_Relational_Data_Sets__In_

Extracting Abstract: Erik_Wilde_Feeds_as_Query_Result_Serializations_In_arXiv_0911__2193__
Saved...dataset!!
